# Step 1: Create Clusters

This notebook implements the chunking, local clustering, cluster merging, and cluster-to-CoFE-category mapping steps of the LLM-based clustering pipeline.

In [ ]:
# © Copyright European Union - 2026

import os
import json
import re
import random
import glob
import pandas as pd
import numpy as np
from collections import Counter
from tqdm import tqdm
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(
    api_key=os.environ["LLM_API_KEY"],
    base_url=os.environ["LLM_BASE_URL"]
)

MODEL_NAME = "llama-3.3-70b-instruct"

random.seed(42)


def extract_json_from_reply(raw_output: str):
    """Extract JSON from LLM reply text."""
    try:
        return json.loads(raw_output)
    except (json.JSONDecodeError, TypeError):
        pass
    raw_output = raw_output.strip()
    if "```json" in raw_output:
        raw_output = raw_output.split("```json")[-1].split("```")[0]
    elif "```" in raw_output:
        parts = raw_output.split("```")
        if len(parts) >= 3:
            raw_output = parts[1]
    # Find JSON structure
    for start_char, end_char in [("[", "]"), ("{", "}")]:
        if start_char in raw_output and end_char in raw_output:
            start = raw_output.index(start_char)
            end = raw_output.rindex(end_char) + 1
            try:
                return json.loads(raw_output[start:end])
            except json.JSONDecodeError:
                continue
    return json.loads(raw_output)


def model_json(system_prompt: str, user_message: str) -> dict:
    """Call LLM and return parsed JSON response."""
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": str(user_message)}
        ],
        temperature=0
    )
    return extract_json_from_reply(response.choices[0].message.content)


def model(system_prompt: str, user_message: str) -> str:
    """Call LLM and return raw text response."""
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": str(user_message)}
        ],
        temperature=0
    )
    return response.choices[0].message.content


In [ ]:
DATA_OUT_BASE_PATH = "../out/"
chunk_clusters_fp = f"{DATA_OUT_BASE_PATH}2.chunk_clusters.json"
merged_clusters_fp = f"{DATA_OUT_BASE_PATH}2.merged_clusters.json"
mapped_clusters_fp = f"{DATA_OUT_BASE_PATH}2.mapped_clusters.json"

In [ ]:
list_proposals = json.load(open("../data/proposals.json"))
random.shuffle(list_proposals)

In [ ]:
print(json.dumps(list_proposals[0], indent=2))

In [ ]:
set([x.get('topictitle', None) for x in list_proposals])

In [ ]:
list_titles = [x["title"] for x in list_proposals]
list_texts = [x["text"] for x in list_proposals]

In [ ]:
from transformers import AutoTokenizer
model_id = "meta-llama/Llama-3.3-70B-Instruct"
#model_id = "google-bert/bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [ ]:
max_tokens = 120000
chunks = []
current_chunk = []
prompt = """The task is to cluster a list of articles.
In order to design the clusters, please consider the most relevant and emerging topics mentioned whithin articles.

Provide in output a JSON with the following structure, do not output anything else.
Before outputing check the JSON is valid. Correct it if it is not

class Cluster(BaseModel):
    name: str # a short name describing the cluster in English
    description_keywords : List[str] # a description of the cluster content in English as a list of multi word expression
    summary: str # redact an executive summary of the information of all the title that belong to that cluster
    
Output:
    List[Cluster] # the list of cluster definition
"""
prompt_tokens = len(tokenizer.encode(prompt, add_special_tokens=True))
current_token_count = prompt_tokens

for text in list_titles:
    # Tokenize the text and count tokens
    tokens = tokenizer.encode(text, add_special_tokens=True)
    text_token_count = len(tokens)

    # Check if adding this text would exceed the limit
    if current_token_count + text_token_count <= max_tokens:
        current_chunk.append(text)
        current_token_count += text_token_count
    else:
        # Start a new chunk if current chunk is not empty
        if current_chunk:
            chunks.append(current_chunk)
        current_chunk = [text]
        current_token_count = text_token_count + prompt_tokens

# Add the last chunk if it's not empty
if current_chunk:
    chunks.append(current_chunk)
    
    
print(len(chunks))
np.mean([len(c) for c in chunks])

### Divide proposals in chunks and build clusters for each chunk

In [ ]:
prompt_cluster = """The task is to cluster a list of articles.
In order to design the clusters, please consider the most relevant and emerging topics mentioned whithin articles.

Provide in output a JSON with the following structure, do not output anything else.
Before outputing check the JSON is valid. Correct it if it is not

class Cluster(BaseModel):
    name: str # a short name describing the cluster in English
    description_keywords : List[str] # a description of the cluster content in English as a list of multi word expression
    summary: str # redact an executive summary of the information of all the title that belong to that cluster
    
Output:
    List[Cluster] # the list of cluster definition
    """

#### Run cluster creation

In [ ]:
def chunked_iterator(l, chunk_size=100, overlap=50):
    """Yield chunks of the list with specified overlap."""
    if overlap >= chunk_size:
        raise ValueError("Overlap must be less than the chunk size.")

    # Iterate over the list in steps of `chunk_size - overlap`
    step = chunk_size - overlap
    for start in range(0, len(l), step):
        if start < len(l)-1 and len(l[start:start + chunk_size]) > overlap:
            yield l[start:start + chunk_size]

In [ ]:
l = [0,1,2,3,4,5,6,7,8,9,10,11]
list(chunked_iterator(l, 5, 2))

In [ ]:
#list_samples = [x for x in chunked_iterator(list_titles, 892, 446)]
#list_samples = [x for x in chunked_iterator(list_titles, 100, 50)]
list_samples = [x for x in chunked_iterator(list_titles, 5000, 2500)]
print(len(list_samples), len(list_samples[-1]))

In [ ]:
# Run clustering detection
list_res = []

for chunk in tqdm(list_samples):
    data = "Articles\n - " + "\n - ".join(chunk)
    #print(prompt_cluster)
    #print(data)
    open("tmp.txt", 'w').write(prompt_cluster + "\n" + data)
    r = model_json(prompt_cluster, data)

    print("=======================")
    print(r)

    list_res.append(r)
json.dump(list_res, open(chunk_clusters_fp, 'w'), indent=2, ensure_ascii=False)

In [ ]:
for r in list_res:
    print("==")
    print([x["name"] for x in r])

### Merge clusters

In [ ]:
merge_size = 10

In [ ]:
prompt_merge = f"""You are provided with a list of cluster names together with their descriptions.
Your task is to merge these list, by merging very similar cluster together.
Propose a final list of {merge_size} clusters. Do not propose code, propose directly the list.

Output a JSON with the following structure, do not output anything else.
Before outputing check the JSON is valid. Correct it if it is not

class Cluster(BaseModel):
    name: str # a short name describing the cluster in English
    description_keywords : List[str] # a description of the cluster content in English as a list of multi word expression
    summary: str # redact an executive summary of the information of all the title that belong to that cluster
    number: int # the ordinal number of the cluster, from 1 to {merge_size}
    
Output:
    List[Cluster] # the list of cluster definition
"""

In [ ]:
list_res = json.load(open(chunk_clusters_fp))
r = model_json(prompt_merge, json.dumps(list_res, indent=2) + f"\n\nPlease provide no more than {merge_size} clusters.")
json.dump(r, open(merged_clusters_fp, 'w'), indent=2, ensure_ascii=False)
print(r)

## Map predicted clusters to original ones

In [ ]:
prompt_map = """Your task is to find the appropriate cluster name of a given list of elements. To find the best cluster, consider the semantically most similar one. The cluster names are the following 10:

%s

Once you find the most semantically similar cluster for the list, output a JSON with the following structure, do not output anything else.
Before providing an output check the JSON is valid. Correct it if it is not

class ClusterMap(BaseModel):
    origianl_cluster_name: str # The original name of the cluster
    mapped_cluster: str # The name of the corresponding cluster
    
Output:
    List[ClusterMap] # the list of cluster mappings
"""

In [ ]:
cofe_definitions = json.load(open("../data/topic_definitions.json"))
for idx, item in enumerate(cofe_definitions.items()):
    prompt_map %= f"{idx}. **{item[0]}**: {item[1]}\n%s"
prompt_map %= ""
prompt_map

In [ ]:
merged_clusters = json.load(open(merged_clusters_fp))
clusters_str = "\n -" + "\n -".join([f"{c['name']}: {c['summary']}" for c in merged_clusters])
r = model_json(prompt_map, clusters_str)
print(r)

In [ ]:
for mapped_cluster in r:
    for predicted_cluster in merged_clusters:
        if predicted_cluster['name'] == mapped_cluster['origianl_cluster_name']:
            predicted_cluster['mapped_cluster'] = mapped_cluster['mapped_cluster']
json.dump(merged_clusters, open(mapped_clusters_fp, 'w'), indent=2, ensure_ascii=False)